<a href="https://colab.research.google.com/github/ypsitau/google-colab/blob/main/create_recognizer_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import google.colab

def generate_model(config_name):
    label_offset = 1 if config_name == 'letters' else 0
    (dataset_train, dataset_test), dataset_info = tfds.load(
        f'emnist/{config_name}',
        split=['train', 'test'],
        as_supervised=True,
        with_info=True,
    )
    num_classes = dataset_info.features['label'].num_classes - label_offset
    #---------------------------------------------------------------------------
    print(f"{config_name}: {num_classes} classes")
    def preprocess(image, label):
        image = tf.transpose(image, perm=[1, 0, 2])
        image = tf.cast(image > 127, tf.float32)
        label = label - label_offset
        return image, label
    dataset_train = dataset_train.map(preprocess).cache().shuffle(10000).batch(64).prefetch(tf.data.AUTOTUNE)
    dataset_test = dataset_test.map(preprocess).batch(64).cache().prefetch(tf.data.AUTOTUNE)
    model = tf.keras.models.Sequential([
        tf.keras.layers.Conv2D(16, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    model.fit(dataset_train, epochs=5, validation_data=dataset_test)
    #---------------------------------------------------------------------------
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    def representative_data_gen():
        for input_value, _ in dataset_test.take(100):
            yield [input_value]
    converter.representative_dataset = representative_data_gen
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    model_tflite = converter.convert()
    filename = f"Recognizer-EMNIST-{config_name}.tflite"
    with open(filename, "wb") as f:
        f.write(model_tflite)
    return filename

filenames = []
filenames.append(generate_model('mnist'))    # 0-9
#filenames.append(generate_model('letters'))  # A-Z, 0-9
#filenames.append(generate_model('bymerge'))  # 0-9, A-Z, a, b, d, e, f, g, h, n, q, r, t
for filename in filenames:
    google.colab.files.download(filename)


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

...

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/emnist/mnist/incomplete.1B9NAE_3.1.0/emnist-test.tfrecord*...:   0%|      …

Dataset emnist downloaded and prepared to /root/tensorflow_datasets/emnist/mnist/3.1.0. Subsequent calls will reuse this data.
mnist: 10 classes


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
644/938 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8795 - loss: 0.3991